In [1]:
"""
Purpose:
    - Create catch probability model using real player trajectories (x, y positions) after throw
    
Output: 
    - Projected player trajectories are saved to the repo: 
        - "config.OUTPUT_DIR / 'model_outputs' / OUTPUT_FILE_NAME"
"""

'\nPurpose:\n    - Create catch probability model using real player trajectories (x, y positions) after throw\n\nOutput: \n    - Projected player trajectories are saved to the repo: \n        - "config.OUTPUT_DIR / \'model_outputs\' / OUTPUT_FILE_NAME"\n'

In [2]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import os
import sys
from typing import List, Tuple, Dict

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

project_root = Path.cwd().parent.parent
print(f"Adding project root to path: {project_root}")
sys.path.insert(0, str(project_root))

# Now import from src.kinematics
from src.kinematics import calculate_speed_and_direction

pd.set_option("display.max_columns", None)

/Users/kniu91/Documents/projects/bdb-26/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Adding project root to path: /Users/kniu91/Documents/projects/bdb-26


In [3]:
# ============================================================================
# CONFIG
# ============================================================================

class Config:
    DATA_DIR = Path("../../data/")
    OUTPUT_DIR = Path("../../outputs")
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    SEED = 44
    N_FOLDS = 5
    BATCH_SIZE = 256
    EPOCHS = 60
    PATIENCE = 30
    LEARNING_RATE = 1e-4
    
    WINDOW_SIZE = 10
    HIDDEN_DIM = 128
    MAX_FUTURE_HORIZON = 94
    
    FIELD_X_MIN, FIELD_X_MAX = 0.0, 120.0
    FIELD_Y_MIN, FIELD_Y_MAX = 0.0, 53.3
    
    K_NEIGH = 6
    RADIUS = 30.0
    TAU = 8.0
    N_ROUTE_CLUSTERS = 7
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(Config.SEED)



In [4]:
"""
train_input :df = Training input data for the Kaggle Prediction competition
    - The info from these frames are supplied as extra info for each play
train_output :df = Training output data for the Kaggle Prediction competition
    - Most importantly, this contains x, y positions of players after throw
    - The info from these frames is used to predict whether a pass is caught or not
traj_output : df = Model output containing predicted x,y positions of players
    - This contains, PREDICTED x, y positions of players after throw
    - The info from these frames is used to predict whether a pass is caught or not

"""
TRAJECTORY_OUTPUT_FILE = "local_submission_cleantest.csv"
config = Config()
config

print("\n[1/4] Loading data...")
train_input_files = [config.DATA_DIR / f"train/input_2023_w{w:02d}.csv" for w in range(1, 19)]
train_output_files = [config.DATA_DIR / f"train/output_2023_w{w:02d}.csv" for w in range(1, 19)]
train_input = pd.concat([pd.read_csv(f) for f in train_input_files if f.exists()])
train_output = pd.concat([pd.read_csv(f) for f in train_output_files if f.exists()])
supplementary_data = pd.read_csv(config.DATA_DIR / "supplementary_data.csv")

print(f"✓ Train input: {train_input.shape}, Train output: {train_output.shape}")
print(f"✓ Train output: {train_output.shape}"
      f"unique plays: {train_output[['game_id','play_id']].drop_duplicates().shape[0]}")
print(f"✓ Supplementary data: {supplementary_data.shape}")

traj_cols = ['game_id', 'play_id', 'nfl_id', 'frame_id', 'pred_x', 'pred_y']
traj_output = pd.read_csv(config.OUTPUT_DIR / 'model_outputs' / TRAJECTORY_OUTPUT_FILE,
                          usecols = traj_cols)
traj_output.rename(columns={'pred_x': 'x', 'pred_y': 'y'}, inplace=True)

train_output.sort_values(by=['game_id', 'play_id', 'nfl_id', 'frame_id'], inplace=True)
traj_output.sort_values(by=['game_id', 'play_id', 'nfl_id', 'frame_id'], inplace=True)

print(f"✓ Projected trajectory output: {traj_output.shape},"
      f"unique plays: {traj_output[['game_id','play_id']].drop_duplicates().shape[0]}")


[1/4] Loading data...


/var/folders/_m/rvnpg_cs6xzcz0vlml0lzkbm0000gp/T/ipykernel_90781/3831966790.py:21: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  supplementary_data = pd.read_csv(config.DATA_DIR / "supplementary_data.csv")


✓ Train input: (4880579, 23), Train output: (562936, 6)
✓ Train output: (562936, 6)unique plays: 14108
✓ Supplementary data: (18009, 41)
✓ Projected trajectory output: (562936, 6),unique plays: 14108


In [5]:
def prepare_completions(suppl_df: pd.DataFrame) -> pd.DataFrame:
    play_results = suppl_df[['game_id','play_id','pass_result']].drop_duplicates()
    play_results.loc[play_results['pass_result'] == 'IN', 'pass_result'] = 'I'
    return play_results

def add_qb_and_ball_angle_features(input_df: pd.DataFrame) -> pd.DataFrame:
    qb_at_throw = (
        input_df[input_df['player_role'] == 'Passer']
        .sort_values(['game_id', 'play_id', 'frame_id'])
        .groupby(['game_id', 'play_id'])
        .last()[['x', 'y']]
        .rename(columns={'x': 'qb_x', 'y': 'qb_y'})
        .reset_index()
    )
    
    input_df = input_df.merge(qb_at_throw, on=['game_id', 'play_id'], how='left')
    dx = input_df['ball_land_x'] - input_df['qb_x']
    dy = input_df['ball_land_y'] - input_df['qb_y']
    ball_angle_rad = np.arctan2(dx, dy)
    input_df['ball_angle_deg'] = np.degrees(ball_angle_rad) % 360
    
    input_df['qb_to_ball_distance'] = np.sqrt(dx**2 + dy**2)
    input_df['ball_speed'] = (input_df['qb_to_ball_distance'] / 
                              (input_df['num_frames_output'] / 10))
    return input_df

def attach_and_prepare_play_level_features(input_df: pd.DataFrame,
                                           output_df:pd.DataFrame,
                                           supplementary_df: pd.DataFrame) -> pd.DataFrame:
    """
    Attaches play-level features from input_df and supplementary_df onto output_df

    Args:
        -input_df (pd.DataFrame): Input DataFrame containing pre-throw tracking data
        -output_df (pd.DataFrame): Output DataFrame containing post-throw tracking data
        -supplementary_df (pd.DataFrame): Supplementary DataFrame containing 
          supplementary play-level information
    """
    play_results = prepare_completions(supplementary_df)
    output_df = calculate_speed_and_direction(output_df)
    input_df = add_qb_and_ball_angle_features(input_df)

    player_level_keys = ["game_id", "play_id", "nfl_id"]
    play_features = ["player_height", "player_weight","player_side",
                     "player_role","player_position",
                      "play_direction", "absolute_yardline_number",
                      "ball_land_x","ball_land_y", "num_frames_output",
                      "ball_angle_deg", "qb_to_ball_distance", "ball_speed"
                ]

    input = input_df[player_level_keys + play_features].drop_duplicates()

    output_df = output_df.merge(input, on=player_level_keys, how='inner')
    output_df = output_df.merge(play_results, on=['game_id','play_id'], 
                                how='left', indicator= True)
    assert all(output_df['_merge'] == 'both')
    output_df = output_df.dropna(subset=['qb_to_ball_distance'])
    output_df = output_df.drop(columns=['_merge'])

    output_df['unique_play_id'] = (output_df['game_id'].astype(str) 
                                   + '_' +
                                     output_df['play_id'].astype(str))
    return output_df


train_output = attach_and_prepare_play_level_features(train_input, 
                                                      train_output, 
                                                      supplementary_data)
traj_output = attach_and_prepare_play_level_features(train_input, 
                                                     traj_output, 
                                                     supplementary_data)

In [6]:
# ============================================================================
# GEOMETRIC BASELINE - THE BREAKTHROUGH
# ============================================================================

def compute_geometric_endpoint(df):
    """
    Compute where each player SHOULD end up based on geometry.
    This is the deterministic part - no learning needed.
    """
    df = df.copy()
    
    # Time to play end
    if 'num_frames_output' in df.columns:
        t_total = df['num_frames_output'] / 10.0
    else:
        t_total = 3.0
    
    df['time_to_endpoint'] = t_total
    
    # Initialize with momentum (default rule)
    df['geo_endpoint_x'] = df['x'] + df['velocity_x'] * t_total
    df['geo_endpoint_y'] = df['y'] + df['velocity_y'] * t_total
    
    # Rule 1: Targeted Receivers converge to ball
    if 'ball_land_x' in df.columns:
        receiver_mask = df['player_role'] == 'Targeted Receiver'
        df.loc[receiver_mask, 'geo_endpoint_x'] = df.loc[receiver_mask, 'ball_land_x']
        df.loc[receiver_mask, 'geo_endpoint_y'] = df.loc[receiver_mask, 'ball_land_y']
        
        # Rule 2: Defenders mirror receivers (maintain offset)
        defender_mask = df['player_role'] == 'Defensive Coverage'
        has_mirror = df.get('mirror_offset_x', 0).notna() & (df.get('mirror_wr_dist', 50) < 15)
        coverage_mask = defender_mask & has_mirror
        
        df.loc[coverage_mask, 'geo_endpoint_x'] = (
            df.loc[coverage_mask, 'ball_land_x'] + 
            df.loc[coverage_mask, 'mirror_offset_x'].fillna(0)
        )
        df.loc[coverage_mask, 'geo_endpoint_y'] = (
            df.loc[coverage_mask, 'ball_land_y'] + 
            df.loc[coverage_mask, 'mirror_offset_y'].fillna(0)
        )
    
    # Clip to field
    df['geo_endpoint_x'] = df['geo_endpoint_x'].clip(Config.FIELD_X_MIN, Config.FIELD_X_MAX)
    df['geo_endpoint_y'] = df['geo_endpoint_y'].clip(Config.FIELD_Y_MIN, Config.FIELD_Y_MAX)
    
    return df

def add_geometric_features(df):
    """Add features that describe the geometric solution"""
    df = compute_geometric_endpoint(df)
    
    # Vector to geometric endpoint
    df['geo_vector_x'] = df['geo_endpoint_x'] - df['x']
    df['geo_vector_y'] = df['geo_endpoint_y'] - df['y']
    df['geo_distance'] = np.sqrt(df['geo_vector_x']**2 + df['geo_vector_y']**2)
    
    # Required velocity to reach geometric endpoint
    t = df['time_to_endpoint'] + 0.1
    df['geo_required_vx'] = df['geo_vector_x'] / t
    df['geo_required_vy'] = df['geo_vector_y'] / t
    
    # Current velocity vs required
    df['geo_velocity_error_x'] = df['geo_required_vx'] - df['velocity_x']
    df['geo_velocity_error_y'] = df['geo_required_vy'] - df['velocity_y']
    df['geo_velocity_error'] = np.sqrt(
        df['geo_velocity_error_x']**2 + df['geo_velocity_error_y']**2
    )
    
    # Required constant acceleration (a = 2*Δx/t²)
    t_sq = t * t
    df['geo_required_ax'] = 2 * df['geo_vector_x'] / t_sq
    df['geo_required_ay'] = 2 * df['geo_vector_y'] / t_sq
    df['geo_required_ax'] = df['geo_required_ax'].clip(-10, 10)
    df['geo_required_ay'] = df['geo_required_ay'].clip(-10, 10)
    
    # Alignment with geometric path
    velocity_mag = np.sqrt(df['velocity_x']**2 + df['velocity_y']**2)
    geo_unit_x = df['geo_vector_x'] / (df['geo_distance'] + 0.1)
    geo_unit_y = df['geo_vector_y'] / (df['geo_distance'] + 0.1)
    df['geo_alignment'] = (
        df['velocity_x'] * geo_unit_x + df['velocity_y'] * geo_unit_y
    ) / (velocity_mag + 0.1)
    
    # Role-specific geometric quality
    df['geo_receiver_urgency'] = df['is_receiver'] * df['geo_distance'] / (t + 0.1)
    df['geo_defender_coupling'] = df['is_coverage'] * (1.0 / (df.get('mirror_wr_dist', 50) + 1.0))
    
    return df


def get_velocity(speed, direction_deg):
    theta = np.deg2rad(direction_deg)
    return speed * np.sin(theta), speed * np.cos(theta)

def height_to_feet(height_str):
    try:
        ft, inches = map(int, str(height_str).split('-'))
        return ft + inches/12
    except:
        return 6.0

def get_opponent_features(input_df: pd.DataFrame) -> pd.DataFrame:
    """Enhanced opponent interaction with MIRROR WR tracking"""
    features = []
    
    for (gid, pid), group in tqdm(input_df.groupby(['game_id', 'play_id']), 
                                   desc="🏈 Opponents", leave=False):
        last = group.sort_values('frame_id').groupby('nfl_id').last()
        
        if len(last) < 2:
            for nid in last.index:
                  features.append({
                      'game_id': gid, 'play_id': pid, 'nfl_id': nid,
                      'nearest_opp_dist': 50.0, 'closing_speed': 0.0,
                      'num_nearby_opp_3': 0, 'num_nearby_opp_5': 0,
                      'mirror_wr_vx': 0.0, 'mirror_wr_vy': 0.0,
                      'mirror_offset_x': 0.0, 'mirror_offset_y': 0.0,
                      'mirror_wr_dist': 50.0,
                  })
            continue
            
        positions = last[['x', 'y']].values
        sides = last['player_side'].values
        speeds = last['s'].values
        directions = last['dir'].values
        roles = last['player_role'].values
        
        receiver_mask = np.isin(roles, ['Targeted Receiver', 'Other Route Runner'])
        
        for i, (nid, side, role) in enumerate(zip(last.index, sides, roles)):
            opp_mask = sides != side
            
            feat = {
                'game_id': gid, 'play_id': pid, 'nfl_id': nid,
                'nearest_opp_dist': 50.0, 'closing_speed': 0.0,
                'num_nearby_opp_3': 0, 'num_nearby_opp_5': 0,
                'mirror_wr_vx': 0.0, 'mirror_wr_vy': 0.0,
                'mirror_offset_x': 0.0, 'mirror_offset_y': 0.0,
                'mirror_wr_dist': 50.0,
            }
            
            if not opp_mask.any():
                features.append(feat)
                continue
            
            opp_positions = positions[opp_mask]
            distances = np.sqrt(((positions[i] - opp_positions)**2).sum(axis=1))
            
            if len(distances) == 0:
                features.append(feat)
                continue
                
            nearest_idx = distances.argmin()
            feat['nearest_opp_dist'] = distances[nearest_idx]
            feat['num_nearby_opp_3'] = (distances < 3.0).sum()
            feat['num_nearby_opp_5'] = (distances < 5.0).sum()
            
            my_vx, my_vy = get_velocity(speeds[i], directions[i])
            opp_speeds = speeds[opp_mask]
            opp_dirs = directions[opp_mask]
            opp_vx, opp_vy = get_velocity(opp_speeds[nearest_idx], opp_dirs[nearest_idx])
            
            rel_vx = my_vx - opp_vx
            rel_vy = my_vy - opp_vy
            to_me = positions[i] - opp_positions[nearest_idx]
            to_me_norm = to_me / (np.linalg.norm(to_me) + 0.1)
            feat['closing_speed'] = -(rel_vx * to_me_norm[0] + rel_vy * to_me_norm[1])
            
            if role == 'Defensive Coverage' and receiver_mask.any():
                rec_positions = positions[receiver_mask]
                rec_distances = np.sqrt(((positions[i] - rec_positions)**2).sum(axis=1))
                
                if len(rec_distances) > 0:
                    closest_rec_idx = rec_distances.argmin()
                    rec_indices = np.where(receiver_mask)[0]
                    actual_rec_idx = rec_indices[closest_rec_idx]
                    
                    rec_vx, rec_vy = get_velocity(speeds[actual_rec_idx], directions[actual_rec_idx])
                    
                    feat['mirror_wr_vx'] = rec_vx
                    feat['mirror_wr_vy'] = rec_vy
                    feat['mirror_wr_dist'] = rec_distances[closest_rec_idx]
                    feat['mirror_offset_x'] = positions[i][0] - rec_positions[closest_rec_idx][0]
                    feat['mirror_offset_y'] = positions[i][1] - rec_positions[closest_rec_idx][1]
            
            features.append(feat)
    
    return pd.DataFrame(features)

def compute_neighbor_embeddings(input_df, k_neigh=Config.K_NEIGH, 
                                radius=Config.RADIUS, tau=Config.TAU, print_logs=True):
    """GNN-lite embeddings"""
    if print_logs:
        print("🕸️  GNN embeddings...")
    
    cols_needed = ["game_id", "play_id", "nfl_id", "frame_id", "x", "y", 
                   "velocity_x", "velocity_y", "player_side"]
    src = input_df[cols_needed].copy()
    
    last = (src.sort_values(["game_id", "play_id", "nfl_id", "frame_id"])
               .groupby(["game_id", "play_id", "nfl_id"], as_index=False)
               .tail(1)
               .rename(columns={"frame_id": "last_frame_id"})
               .reset_index(drop=True))
    
    all_players = last[["game_id", "play_id", "nfl_id"]].copy()

    tmp = last.merge(
        src.rename(columns={
            "frame_id": "nb_frame_id", "nfl_id": "nfl_id_nb",
            "x": "x_nb", "y": "y_nb", 
            "velocity_x": "vx_nb", "velocity_y": "vy_nb", 
            "player_side": "player_side_nb"
        }),
        left_on=["game_id", "play_id", "last_frame_id"],
        right_on=["game_id", "play_id", "nb_frame_id"],
        how="left"
    )
    
    tmp = tmp[tmp["nfl_id_nb"] != tmp["nfl_id"]]
    tmp["dx"] = tmp["x_nb"] - tmp["x"]
    tmp["dy"] = tmp["y_nb"] - tmp["y"]
    tmp["dvx"] = tmp["vx_nb"] - tmp["velocity_x"]
    tmp["dvy"] = tmp["vy_nb"] - tmp["velocity_y"]
    tmp["dist"] = np.sqrt(tmp["dx"]**2 + tmp["dy"]**2)
    
    tmp = tmp[np.isfinite(tmp["dist"]) & (tmp["dist"] > 1e-6)]
    if radius is not None:
        tmp = tmp[tmp["dist"] <= radius]
    
    tmp["is_ally"] = (tmp["player_side_nb"] == tmp["player_side"]).astype(np.float32)
    
    keys = ["game_id", "play_id", "nfl_id"]
    tmp["rnk"] = tmp.groupby(keys)["dist"].rank(method="first")
    if k_neigh is not None:
        tmp = tmp[tmp["rnk"] <= float(k_neigh)]
    
    tmp["w"] = np.exp(-tmp["dist"] / float(tau))
    sum_w = tmp.groupby(keys)["w"].transform("sum")
    tmp["wn"] = np.where(sum_w > 0, tmp["w"] / sum_w, 0.0)
    
    tmp["wn_ally"] = tmp["wn"] * tmp["is_ally"]
    tmp["wn_opp"] = tmp["wn"] * (1.0 - tmp["is_ally"])
    
    for col in ["dx", "dy", "dvx", "dvy"]:
        tmp[f"{col}_ally_w"] = tmp[col] * tmp["wn_ally"]
        tmp[f"{col}_opp_w"] = tmp[col] * tmp["wn_opp"]
    
    tmp["dist_ally"] = np.where(tmp["is_ally"] > 0.5, tmp["dist"], np.nan)
    tmp["dist_opp"] = np.where(tmp["is_ally"] < 0.5, tmp["dist"], np.nan)
    
    ag = tmp.groupby(keys).agg(
        gnn_ally_dx_mean=("dx_ally_w", "sum"),
        gnn_ally_dy_mean=("dy_ally_w", "sum"),
        gnn_ally_dvx_mean=("dvx_ally_w", "sum"),
        gnn_ally_dvy_mean=("dvy_ally_w", "sum"),
        gnn_opp_dx_mean=("dx_opp_w", "sum"),
        gnn_opp_dy_mean=("dy_opp_w", "sum"),
        gnn_opp_dvx_mean=("dvx_opp_w", "sum"),
        gnn_opp_dvy_mean=("dvy_opp_w", "sum"),
        gnn_ally_cnt=("is_ally", "sum"),
        gnn_opp_cnt=("is_ally", lambda s: float(len(s) - s.sum())),
        gnn_ally_dmin=("dist_ally", "min"),
        gnn_ally_dmean=("dist_ally", "mean"),
        gnn_opp_dmin=("dist_opp", "min"),
        gnn_opp_dmean=("dist_opp", "mean"),
    ).reset_index()

    ag = all_players.merge(ag, on=keys, how="left")

    
    near = tmp.loc[tmp["rnk"] <= 3, keys + ["rnk", "dist"]].copy()
    if len(near) > 0:
        near["rnk"] = near["rnk"].astype(int)
        dwide = near.pivot_table(index=keys, columns="rnk", values="dist", aggfunc="first")
        dwide = dwide.rename(columns={1: "gnn_d1", 2: "gnn_d2", 3: "gnn_d3"}).reset_index()
        ag = ag.merge(dwide, on=keys, how="left")
    
    for c in ["gnn_ally_dx_mean", "gnn_ally_dy_mean", "gnn_ally_dvx_mean", "gnn_ally_dvy_mean",
              "gnn_opp_dx_mean", "gnn_opp_dy_mean", "gnn_opp_dvx_mean", "gnn_opp_dvy_mean"]:
        ag[c] = ag[c].fillna(0.0)
    for c in ["gnn_ally_cnt", "gnn_opp_cnt"]:
        ag[c] = ag[c].fillna(0.0)
    for c in ["gnn_ally_dmin", "gnn_opp_dmin", "gnn_ally_dmean", "gnn_opp_dmean", 
              "gnn_d1", "gnn_d2", "gnn_d3"]:
        if c in ag.columns:
            ag[c] = ag[c].fillna(radius if radius is not None else 30.0)
        else:
            # Create missing columns with default values
            ag[c] = radius if radius is not None else 30.0
  
    return ag

def prepare_sequences_geometric(input_df, 
                                window_size=5,
                                print_logs=True
                                ) -> Tuple[List[np.ndarray], List[int], List[Dict[str,int]], List[str]]:
    """
    YOUR 154 features + 13 geometric features = 167 total
    
    Why is "is_training" set to true? Can we return what we need 
    without it?
    """
    if print_logs:
        print(f"\n{'='*80}")
        print(f"PREPARING GEOMETRIC SEQUENCES")
        print(f"{'='*80}")
    
    input_df = input_df.copy()
    input_df = input_df.sort_values(['game_id', 'play_id', 'nfl_id', 'frame_id'])
    
    if print_logs:
        print("Step 1: Base features...")
    
    input_df['player_height_feet'] = input_df['player_height'].apply(height_to_feet)
    height_parts = input_df['player_height'].str.split('-', expand=True)
    input_df['height_inches'] = height_parts[0].astype(float) * 12 + height_parts[1].astype(float)
    input_df['bmi'] = (input_df['player_weight'] / (input_df['height_inches']**2)) * 703
    
    dir_rad = np.deg2rad(input_df['dir'].fillna(0))
    input_df['velocity_x'] = input_df['s'] * np.sin(dir_rad)
    input_df['velocity_y'] = input_df['s'] * np.cos(dir_rad)
    # input_df['acceleration_x'] = input_df['a'] * np.cos(dir_rad)
    # input_df['acceleration_y'] = input_df['a'] * np.sin(dir_rad)
    
    input_df['speed_squared'] = input_df['s'] ** 2
    # input_df['accel_magnitude'] = np.sqrt(input_df['acceleration_x']**2 + input_df['acceleration_y']**2)
    input_df['momentum_x'] = input_df['velocity_x'] * input_df['player_weight']
    input_df['momentum_y'] = input_df['velocity_y'] * input_df['player_weight']
    input_df['kinetic_energy'] = 0.5 * input_df['player_weight'] * input_df['speed_squared']
    
    # input_df['orientation_diff'] = np.abs(input_df['o'] - input_df['dir'])
    # input_df['orientation_diff'] = np.minimum(input_df['orientation_diff'], 360 - input_df['orientation_diff'])
    
    input_df['is_offense'] = (input_df['player_side'] == 'Offense').astype(int)
    input_df['is_defense'] = (input_df['player_side'] == 'Defense').astype(int)
    input_df['is_receiver'] = (input_df['player_role'] == 'Targeted Receiver').astype(int)
    input_df['is_coverage'] = (input_df['player_role'] == 'Defensive Coverage').astype(int)
    input_df['is_passer'] = (input_df['player_role'] == 'Passer').astype(int)
    input_df['role_targeted_receiver'] = input_df['is_receiver']
    input_df['role_defensive_coverage'] = input_df['is_coverage']
    input_df['role_passer'] = input_df['is_passer']
    input_df['side_offense'] = input_df['is_offense']
    
    if 'ball_land_x' in input_df.columns:
        ball_dx = input_df['ball_land_x'] - input_df['x']
        ball_dy = input_df['ball_land_y'] - input_df['y']
        input_df['distance_to_ball'] = np.sqrt(ball_dx**2 + ball_dy**2)
        input_df['dist_to_ball'] = input_df['distance_to_ball']
        input_df['dist_squared'] = input_df['distance_to_ball'] ** 2
        input_df['angle_to_ball'] = np.arctan2(ball_dy, ball_dx)
        input_df['ball_direction_x'] = ball_dx / (input_df['distance_to_ball'] + 1e-6)
        input_df['ball_direction_y'] = ball_dy / (input_df['distance_to_ball'] + 1e-6)
        input_df['closing_speed_ball'] = (
            input_df['velocity_x'] * input_df['ball_direction_x'] +
            input_df['velocity_y'] * input_df['ball_direction_y']
        )
        input_df['velocity_toward_ball'] = (
            input_df['velocity_x'] * np.cos(input_df['angle_to_ball']) + 
            input_df['velocity_y'] * np.sin(input_df['angle_to_ball'])
        )
        input_df['velocity_alignment'] = np.cos(input_df['angle_to_ball'] - dir_rad)
        # input_df['angle_diff'] = np.abs(input_df['o'] - np.degrees(input_df['angle_to_ball']))
        # input_df['angle_diff'] = np.minimum(input_df['angle_diff'], 360 - input_df['angle_diff'])
    
    if print_logs:
        print("Step 2: Advanced features...")
    
    opp_features = get_opponent_features(input_df)
    input_df = input_df.merge(opp_features, on=['game_id', 'play_id', 'nfl_id'], how='left')
    
    # if is_training:
    #     route_features, route_kmeans, route_scaler = extract_route_patterns(input_df)
    # else:
    #     route_features = extract_route_patterns(input_df, route_kmeans, route_scaler, fit=False)
    # input_df = input_df.merge(route_features, on=['game_id', 'play_id', 'nfl_id'], how='left')
    
    gnn_features = compute_neighbor_embeddings(input_df, print_logs = print_logs)
    input_df = input_df.merge(gnn_features, on=['game_id', 'play_id', 'nfl_id'], how='left')
    
    if 'nearest_opp_dist' in input_df.columns:
        input_df['pressure'] = 1 / np.maximum(input_df['nearest_opp_dist'], 0.5)
        input_df['under_pressure'] = (input_df['nearest_opp_dist'] < 3).astype(int)
        input_df['pressure_x_speed'] = input_df['pressure'] * input_df['s']
    
    if 'mirror_wr_vx' in input_df.columns:
        s_safe = np.maximum(input_df['s'], 0.1)
        input_df['mirror_similarity'] = (
            input_df['velocity_x'] * input_df['mirror_wr_vx'] + 
            input_df['velocity_y'] * input_df['mirror_wr_vy']
        ) / s_safe
        input_df['mirror_offset_dist'] = np.sqrt(
            input_df['mirror_offset_x']**2 + input_df['mirror_offset_y']**2
        )
        input_df['mirror_alignment'] = input_df['mirror_similarity'] * input_df['role_defensive_coverage']
    
    if print_logs:
        print("Step 3: Temporal features...")
    
    gcols = ['game_id', 'play_id', 'nfl_id']
    
    # TODO: Add more lags/windows as needed?
    for lag in [1, 2, 3, 4, 5]:
        # for col in ['x', 'y', 'velocity_x', 'velocity_y', 's', 'a']:
        for col in ['x', 'y', 'velocity_x', 'velocity_y', 's']:
            if col in input_df.columns:
                input_df[f'{col}_lag{lag}'] = input_df.groupby(gcols)[col].shift(lag)
    
    for window in [3, 5]:
        for col in ['x', 'y', 'velocity_x', 'velocity_y', 's']:
            if col in input_df.columns:
                input_df[f'{col}_rolling_mean_{window}'] = (
                    input_df.groupby(gcols)[col]
                      .rolling(window, min_periods=1).mean()
                      .reset_index(level=[0,1,2], drop=True)
                )
                input_df[f'{col}_rolling_std_{window}'] = (
                    input_df.groupby(gcols)[col]
                      .rolling(window, min_periods=1).std()
                      .reset_index(level=[0,1,2], drop=True)
                )
    
    for col in ['velocity_x', 'velocity_y']:
        if col in input_df.columns:
            input_df[f'{col}_delta'] = input_df.groupby(gcols)[col].diff()
    
    input_df['velocity_x_ema'] = input_df.groupby(gcols)['velocity_x'].transform(
        lambda x: x.ewm(alpha=0.3, adjust=False).mean()
    )
    input_df['velocity_y_ema'] = input_df.groupby(gcols)['velocity_y'].transform(
        lambda x: x.ewm(alpha=0.3, adjust=False).mean()
    )
    input_df['speed_ema'] = input_df.groupby(gcols)['s'].transform(
        lambda x: x.ewm(alpha=0.3, adjust=False).mean()
    )
    
    if print_logs:
        print("Step 4: Time features...")
    
    if 'num_frames_output' in input_df.columns:
        max_frames = input_df['num_frames_output']
        
        input_df['max_play_duration'] = max_frames / 10.0
        input_df['frame_time'] = input_df['frame_id'] / 10.0
        input_df['progress_ratio'] = input_df['frame_id'] / np.maximum(max_frames, 1)
        input_df['time_remaining'] = (max_frames - input_df['frame_id']) / 10.0
        input_df['frames_remaining'] = max_frames - input_df['frame_id']
        
        input_df['expected_x_at_ball'] = input_df['x'] + input_df['velocity_x'] * input_df['frame_time']
        input_df['expected_y_at_ball'] = input_df['y'] + input_df['velocity_y'] * input_df['frame_time']
        
        if 'ball_land_x' in input_df.columns:
            input_df['error_from_ball_x'] = input_df['expected_x_at_ball'] - input_df['ball_land_x']
            input_df['error_from_ball_y'] = input_df['expected_y_at_ball'] - input_df['ball_land_y']
            input_df['error_from_ball'] = np.sqrt(
                input_df['error_from_ball_x']**2 + input_df['error_from_ball_y']**2
            )
            
            input_df['weighted_dist_by_time'] = input_df['dist_to_ball'] / (input_df['frame_time'] + 0.1)
            input_df['dist_scaled_by_progress'] = input_df['dist_to_ball'] * (1 - input_df['progress_ratio'])
        
        input_df['time_squared'] = input_df['frame_time'] ** 2
        input_df['velocity_x_progress'] = input_df['velocity_x'] * input_df['progress_ratio']
        input_df['velocity_y_progress'] = input_df['velocity_y'] * input_df['progress_ratio']
        input_df['speed_scaled_by_time_left'] = input_df['s'] * input_df['time_remaining']
        
        input_df['actual_play_length'] = max_frames
        input_df['length_ratio'] = max_frames / 30.0
    
    # 🎯 THE BREAKTHROUGH: Add geometric features
    if print_logs:
        print("Step 5: 🎯 Geometric endpoint features...")
    input_df = add_geometric_features(input_df)
    
    if print_logs:
        print("Step 6: Building feature list...")
    
    # Your 154 proven features
    feature_cols = [
        'x', 'y', 's', 
        # 'a', 'o', 
        'dir', 'frame_id', 'ball_land_x', 'ball_land_y',
        'player_height_feet', 'player_weight', 'height_inches', 'bmi',
        'velocity_x', 'velocity_y', 
        # 'acceleration_x', 'acceleration_y',
        'momentum_x', 'momentum_y', 'kinetic_energy',
        'speed_squared', 'accel_magnitude', 
        # 'orientation_diff',
        'is_offense', 'is_defense', 'is_receiver', 'is_coverage', 'is_passer',
        'role_targeted_receiver', 'role_defensive_coverage', 'role_passer', 'side_offense',
        'distance_to_ball', 'dist_to_ball', 'dist_squared', 'angle_to_ball', 
        'ball_direction_x', 'ball_direction_y', 'closing_speed_ball',
        'velocity_toward_ball', 'velocity_alignment', 
        # 'angle_diff',
        'nearest_opp_dist', 'closing_speed', 'num_nearby_opp_3', 'num_nearby_opp_5',
        'mirror_wr_vx', 'mirror_wr_vy', 'mirror_offset_x', 'mirror_offset_y',
        'pressure', 'under_pressure', 'pressure_x_speed', 
        'mirror_similarity', 'mirror_offset_dist', 'mirror_alignment',
        # 'route_pattern', 'traj_straightness', 'traj_max_turn', 'traj_mean_turn',
        # 'traj_depth', 'traj_width', 'speed_mean', 'speed_change',
        'gnn_ally_dx_mean', 'gnn_ally_dy_mean', 'gnn_ally_dvx_mean', 'gnn_ally_dvy_mean',
        'gnn_opp_dx_mean', 'gnn_opp_dy_mean', 'gnn_opp_dvx_mean', 'gnn_opp_dvy_mean',
        'gnn_ally_cnt', 'gnn_opp_cnt',
        'gnn_ally_dmin', 'gnn_ally_dmean', 'gnn_opp_dmin', 'gnn_opp_dmean',
        'gnn_d1', 'gnn_d2', 'gnn_d3',
        'ball_angle_deg', 'qb_to_ball_distance', 'ball_speed',
    ]
    
    for lag in [1, 2, 3, 4, 5]:
        # for col in ['x', 'y', 'velocity_x', 'velocity_y', 's', 'a']:
        for col in ['x', 'y', 'velocity_x', 'velocity_y', 's']:
            feature_cols.append(f'{col}_lag{lag}')
    
    for window in [3, 5]:
        for col in ['x', 'y', 'velocity_x', 'velocity_y', 's']:
            feature_cols.append(f'{col}_rolling_mean_{window}')
            feature_cols.append(f'{col}_rolling_std_{window}')
    
    feature_cols.extend(['velocity_x_delta', 'velocity_y_delta'])
    feature_cols.extend(['velocity_x_ema', 'velocity_y_ema', 'speed_ema'])
    
    feature_cols.extend([
        'max_play_duration', 'frame_time', 'progress_ratio', 'time_remaining', 'frames_remaining',
        'expected_x_at_ball', 'expected_y_at_ball', 
        'error_from_ball_x', 'error_from_ball_y', 'error_from_ball',
        'time_squared', 'weighted_dist_by_time', 
        'velocity_x_progress', 'velocity_y_progress', 'dist_scaled_by_progress',
        'speed_scaled_by_time_left', 'actual_play_length', 'length_ratio',
    ])
    
    # 🎯 Add 13 geometric features
    feature_cols.extend([
        'geo_endpoint_x', 'geo_endpoint_y',
        'geo_vector_x', 'geo_vector_y', 'geo_distance',
        'geo_required_vx', 'geo_required_vy',
        'geo_velocity_error_x', 'geo_velocity_error_y', 'geo_velocity_error',
        'geo_required_ax', 'geo_required_ay',
        'geo_alignment',
    ])
    
    feature_cols = [c for c in feature_cols if c in input_df.columns]
    if print_logs:
        print(f"✓ Using {len(feature_cols)} features ({len(feature_cols) - 13} proven + 13 geometric)")
    
        print("Step 7: Creating sequences...")
    
    target_rows = input_df.copy() # Instantiate before we mess with input_df
    target_groups = target_rows[['game_id', 'play_id']].drop_duplicates()

    sequences, targets_catch, sequence_ids = [], [], []

    for _, row in tqdm(target_groups.iterrows(), total=len(target_groups), desc="Creating sequences", disable = not print_logs):
        # key = (row['game_id'], row['play_id'], row['nfl_id']) 
        key = (row['game_id'], row['play_id'])
        
        try:
            group_df = input_df[(input_df['game_id']==row['game_id']) &
                                 (input_df['play_id']==row['play_id'])]
        except KeyError:
            continue
        
        group_df = group_df[group_df['player_role']=='Targeted Receiver']
        input_window = group_df.tail(window_size)
        
        if len(input_window) < window_size:
            pad_len = window_size - len(input_window)
            pad_df = pd.DataFrame(np.nan, index=range(pad_len), columns=input_window.columns)
            input_window = pd.concat([pad_df, input_window], ignore_index=True)
        
        input_window = input_window.fillna(group_df.mean(numeric_only=True))
        seq = input_window[feature_cols].values
        
        if np.isnan(seq).any():
            # if is_training:
            #     # Print the columns that have NaNs
            #     nan_cols = input_window[feature_cols].columns[input_window[feature_cols].isna().any()].tolist()
            #     print(f"Columns with NaNs: {nan_cols}")
            #     print()
            #     continue
            seq = np.nan_to_num(seq, nan=0.0)
        
        sequences.append(seq)
        
        # Store geometric endpoint for this player
        geo_x = input_window.iloc[-1]['geo_endpoint_x']
        geo_y = input_window.iloc[-1]['geo_endpoint_y']
        
        out_grp = input_df[
            (input_df['game_id']==group_df.iloc[0]['game_id']) &
            (input_df['play_id']==group_df.iloc[0]['play_id']) &
            (input_df['nfl_id']==group_df.iloc[0]['nfl_id'])
        ].sort_values('frame_id')
        
        was_catch = out_grp['pass_result'].values[0] == 'C'
        targets_catch.append(1 if was_catch else 0)
        
        sequence_ids.append({
            'game_id': key[0],
            'play_id': key[1],
            'frame_id': input_window.iloc[-1]['frame_id']
        })

    if print_logs:
        print(f"✓ Created {len(sequences)} sequences")
    
    return (sequences, 
            targets_catch,
            # targets_dx,
            # targets_dy, 
            # targets_frame_ids, 
            sequence_ids, 
            # geo_endpoints_x, 
            # geo_endpoints_y, 
            # route_kmeans, 
            # route_scaler,
            feature_cols)
    # return sequences, sequence_ids#, geo_endpoints_x, geo_endpoints_y
    # return input_df

In [12]:
from dataclasses import dataclass

sample_plays = False
number_sampled = 1000

min_frame_length = 7
if min_frame_length:
    train_input = train_input[train_input['num_frames_output'] >= min_frame_length].copy()    

if sample_plays:
    unique_plays = train_input[['game_id', 'play_id']].drop_duplicates()
    sampled_plays = unique_plays.sample(n=number_sampled, random_state=42)
else:
    sampled_plays = train_input[['game_id', 'play_id']].drop_duplicates()

print(f"Using {len(sampled_plays)} plays for debugging/testing.") 
print("Note this doesn't count any samples dropped from train_output during data processing")

@dataclass
class TrajectoryObject:
    input_df: pd.DataFrame
    sequences: List[np.ndarray]
    targets_catch: List[int]
    sequence_ids: List[dict]
    feature_cols: List[str]

real_output_sampled = train_output.merge(sampled_plays, 
                                         on=['game_id', 'play_id'], 
                                         how='inner')
real_result: Tuple[List[np.ndarray], List[Dict[str, int]]]
real_result = prepare_sequences_geometric(real_output_sampled)
realTrajectoryObject = TrajectoryObject(real_output_sampled, *real_result)

traj_output_sampled = traj_output.merge(sampled_plays, 
                                        on=['game_id', 'play_id'], 
                                        how='inner')
traj_result: Tuple[List[np.ndarray], List[Dict[str, int]]]
traj_result = prepare_sequences_geometric(traj_output_sampled)
projTrajectoryObject = TrajectoryObject(
    traj_output_sampled, *traj_result
)


Using 13135 plays for debugging/testing.
Note this doesn't count any samples dropped from train_output during data processing

PREPARING GEOMETRIC SEQUENCES
Step 1: Base features...
Step 2: Advanced features...


🕸️  GNN embeddings...
Step 3: Temporal features...
Step 4: Time features...
Step 5: 🎯 Geometric endpoint features...
Step 6: Building feature list...
✓ Using 150 features (137 proven + 13 geometric)
Step 7: Creating sequences...


Creating sequences: 100%|██████████| 13132/13132 [02:23<00:00, 91.48it/s]


✓ Created 13132 sequences

PREPARING GEOMETRIC SEQUENCES
Step 1: Base features...
Step 2: Advanced features...


🕸️  GNN embeddings...
Step 3: Temporal features...
Step 4: Time features...
Step 5: 🎯 Geometric endpoint features...
Step 6: Building feature list...
✓ Using 150 features (137 proven + 13 geometric)
Step 7: Creating sequences...


Creating sequences: 100%|██████████| 13132/13132 [02:27<00:00, 89.29it/s]


✓ Created 13132 sequences


In [13]:
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score

def compute_metrics(y_true: np.ndarray, y_pred_proba: np.ndarray, prefix: str = "") -> dict:
    """
    Compute classification metrics.
    
    Args:
        y_true: Ground truth labels (0 or 1)
        y_pred_proba: Predicted probabilities (0 to 1)
        prefix: Optional prefix for metric names (e.g., "val_", "real_")
    
    Returns:
        Dictionary of metrics
    """
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    metrics = {
        f'{prefix}bce': log_loss(y_true, y_pred_proba),
        f'{prefix}auc': roc_auc_score(y_true, y_pred_proba),
        f'{prefix}acc': accuracy_score(y_true, y_pred),
    }
    
    return metrics

def print_metrics(metrics: dict, title: str = "Metrics"):
    """Pretty print metrics"""
    print(f"\n{title}:")
    for name, value in metrics.items():
        print(f"  {name}: {value:.4f}")

In [ ]:
def create_hybrid_trajectory(base_df: pd.DataFrame, 
                             swap_df: pd.DataFrame, 
                             game_id: int, 
                             play_id: int, 
                             swap_nfl_id: int) -> pd.DataFrame:
    """
    Create hybrid trajectory:
        -Use this to swap in individual players' real/projected trajectories into a base 
        that uses other players' projected/real trajectory.
        -So for any case, we're either swapping in a player's real trajectory into a baseline
        with other players' projected trajectories, or vice versa.
    
    Args:
        base_df: The baseline trajectory df w. trajectories of all non-swapped players
        swap_df: The trajectory df containing swapped player's trajectory
        game_id, play_id: Identify the play
        swap_nfl_id: Which defender to use real trajectory for
    
    Returns:
        Hybrid DataFrame ready for feature engineering
    """
    # Start with projected
    play_proj = base_df[(base_df['game_id'] == game_id) & 
                        (base_df['play_id'] == play_id)].copy()
    
    # Get real trajectory for ONE defender
    defender_real = swap_df[(swap_df['game_id'] == game_id) & 
                           (swap_df['play_id'] == play_id) & 
                           (swap_df['nfl_id'] == swap_nfl_id)].copy()
    
    # Swap: remove projected version, add real version
    play_hybrid = play_proj[play_proj['nfl_id'] != swap_nfl_id]
    play_hybrid = pd.concat([play_hybrid, defender_real], ignore_index=True)
    play_hybrid = play_hybrid.sort_values(['nfl_id', 'frame_id'])
    
    return play_hybrid

def get_defender_impact(model: nn.Module, 
                        scaler: any, 
                        proj_df: pd.DataFrame, 
                        real_df: pd.DataFrame, 
                        game_id: int, 
                        play_id: int, 
                        config: any) -> pd.DataFrame:
    """
    Measure each defender's impact on catch probability.
    
    Returns:
        DataFrame with columns: [nfl_id, baseline_prob, real_prob, delta]
    """
    # Baseline: All projected
    play_proj = proj_df[(proj_df['game_id'] == game_id) & 
                        (proj_df['play_id'] == play_id)]
    
    # Get list of defenders
    all_player_ids = play_proj['nfl_id'].unique()
    if len(all_player_ids) == 0:
        return None
    
    # Baseline prediction (all projected)
    baseline_seq, _, _, _ = prepare_sequences_geometric(
        play_proj, window_size=5, print_logs = False
    )
    baseline_seq_sc = [scaler.transform(s) for s in baseline_seq]
    baseline_flat = np.vstack([s.flatten() for s in baseline_seq_sc])
    baseline_prob = model.predict_proba(baseline_flat)[:, 1][0]
    
    results = []
    for single_player_id in all_player_ids:
        # print(f"Processing for defender {defender_id}")
        # Create hybrid: this defender real, others projected
        hybrid_df = create_hybrid_trajectory(
            proj_df, real_df, game_id, play_id, single_player_id
        )
        
        # Re-engineer features (GNN, opponents, etc.)
        hybrid_seq, _, _, _ = prepare_sequences_geometric(
            hybrid_df, window_size=5, print_logs = False
        )
        
        # Predict
        hybrid_seq_sc = [scaler.transform(s) for s in hybrid_seq]
        hybrid_flat = np.vstack([s.flatten() for s in hybrid_seq_sc])
        real_prob = model.predict_proba(hybrid_flat)[:, 1][0]
        
        results.append({
            'nfl_id': single_player_id,
            'player_role': play_proj[play_proj['nfl_id'] == single_player_id]['player_role'].values[0],
            'baseline_prob': baseline_prob,
            'real_prob': real_prob,
            'delta': real_prob - baseline_prob,  # Negative = defender suppressed catch
        })
    
    return pd.DataFrame(results)

def get_defender_impact_flipped(model: nn.Module, 
                        scaler: any, 
                        proj_df: pd.DataFrame, 
                        real_df: pd.DataFrame, 
                        game_id: int, 
                        play_id: int, 
                        config: any) -> pd.DataFrame:
    """
    Measure each defender's impact on catch probability.

    In this case, the baseline is all REAL, and we swap in projected for each defender.
    
    Returns:
        DataFrame with columns: [nfl_id, baseline_prob, real_prob, delta]
    """
    # Baseline: All projected
    play_real = real_df[(real_df['game_id'] == game_id) & 
                        (real_df['play_id'] == play_id)]
    
    
    # Get list of defenders
    all_player_ids = play_real['nfl_id'].unique()
    if len(all_player_ids) == 0:
        return None
    
    # Baseline prediction (all projected)
    real_seq, _, _, _ = prepare_sequences_geometric(
        play_real, window_size=5, print_logs = False
    )
    real_seq_sc = [scaler.transform(s) for s in real_seq]
    real_flat = np.vstack([s.flatten() for s in real_seq_sc])
    real_prob = model.predict(real_flat)[0]

    
    results = []
    for single_player_id in all_player_ids:
        # Create hybrid: this defender real, others projected
        hybrid_df = create_hybrid_trajectory(
            real_df, proj_df, game_id, play_id, single_player_id
        )
        
        # Re-engineer features (GNN, opponents, etc.)
        hybrid_seq, _, _, _ = prepare_sequences_geometric(
            hybrid_df, window_size=5, print_logs = False
        )
        
        # Predict
        hybrid_seq_sc = [scaler.transform(s) for s in hybrid_seq]
        hybrid_flat = np.vstack([s.flatten() for s in hybrid_seq_sc])
        baseline_prob = model.predict(hybrid_flat)[0]  
        # NOTE: Baseline here is the swapped projected version
        
        results.append({
            'nfl_id': single_player_id,
            'player_role': play_real[play_real['nfl_id'] == single_player_id]['player_role'].values[0],
            'baseline_prob': baseline_prob,
            'real_prob': real_prob,
            'delta': real_prob - baseline_prob,  # Negative = defender suppressed catch
        })
    
    return pd.DataFrame(results)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV

def train_lightgbm_with_tuning(X_train: List[np.ndarray], 
                                y_train: List[int],
                                X_val: List[np.ndarray],
                                y_val: List[int],
                                config: Config):
    """
    Train LightGBM with GridSearchCV hyperparameter optimization.
    """
    # Flatten sequences
    X_train_flat = np.vstack([s.flatten() for s in X_train])
    X_val_flat = np.vstack([s.flatten() for s in X_val])
    y_train_array = np.array(y_train)
    y_val_array = np.array(y_val)
    
    print(f"  Training: {X_train_flat.shape} | Validation: {X_val_flat.shape}")
    
    param_grid = {
            'num_leaves': [15],  # 31=conservative, 63=current, 127=aggressive
            'learning_rate': [0.0075],  # 0.05=current, test slower/faster
            'reg_lambda': [2.0],  # L2 penalty - you had 0, try stronger
            
            'n_estimators': [500],
            'min_child_samples': [20],
            'subsample': [0.8],
            'colsample_bytree': [0.8],
            'reg_alpha': [0],
        }
    base_model = lgb.LGBMClassifier(
        objective='binary',
        metric='binary_logloss',
        random_state=config.SEED,
        n_jobs=-1,
        verbose=-1
    )
    
    grid_search = GridSearchCV(
        base_model, param_grid, cv=3, scoring='neg_log_loss', n_jobs=-1, verbose=1
    )
    
    print("  ⚡ Running GridSearchCV...")
    grid_search.fit(X_train_flat, y_train_array)
    
    print(f"  ✓ Best CV AUC: {grid_search.best_score_:.4f}")
    print(f"  ✓ Best params: {grid_search.best_params_}")
    
    # Convert to LightGBM native params
    best_params = grid_search.best_estimator_.get_params()
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'seed': config.SEED,
        **{k: best_params[k] for k in ['num_leaves', 'learning_rate', 'n_estimators', 
                                        'min_child_samples', 'subsample', 'colsample_bytree', 
                                        'reg_alpha', 'reg_lambda']}
    }

    # Train final model with early stopping
    train_data = lgb.Dataset(X_train_flat, label=y_train_array)
    val_data = lgb.Dataset(X_val_flat, label=y_val_array, reference=train_data)
    
    final_model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=1000,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=500)
        ]
    )
    
    # Evaluate
    y_train_pred = final_model.predict(X_train_flat)
    y_val_pred = final_model.predict(X_val_flat)
    
    train_auc = roc_auc_score(y_train_array, y_train_pred)
    val_auc = roc_auc_score(y_val_array, y_val_pred)
    train_bce = log_loss(y_train_array, y_train_pred)
    val_bce = log_loss(y_val_array, y_val_pred)
    
    print(f"\n  📊 Final Metrics:")
    print(f"     Train AUC: {train_auc:.4f} | BCE: {train_bce:.4f}")
    print(f"     Val   AUC: {val_auc:.4f} | BCE: {val_bce:.4f}")

    if train_auc - val_auc > 0.10:
        print(f"  ⚠️  WARNING: AUC gap = {train_auc - val_auc:.3f} (overfitting)")
    if train_bce < val_bce - 0.05:
        print(f"  ⚠️  WARNING: BCE gap = {val_bce - train_bce:.3f} (poor calibration)")
    
    return final_model, val_auc, val_bce


In [ ]:
from sklearn.metrics import roc_auc_score, log_loss

# ============================================================================
# TRAINING LOOP
# ============================================================================

CATCH_PROB_OUTPUT_FILE = 'catch_probabilities_lgb.csv'
DEFENDER_IMPACT_OUTPUT_FILE = 'defender_impact_lgb.csv'
CALCULATE_DEFENDER_IMPACT : bool= True

print("\n[3/4] Training LightGBM models...")
sequence_ids = realTrajectoryObject.sequence_ids
sequences = realTrajectoryObject.sequences

groups = np.array([d['game_id'] for d in sequence_ids])
gkf = GroupKFold(n_splits=config.N_FOLDS)

models, scalers, fold_metrics = [], [], []

want_out_fold_metrics = True

for fold, (tr, va) in enumerate(gkf.split(sequences, groups=groups), 1):
    print(f"\n{'='*60}")
    print(f"Fold {fold}/{config.N_FOLDS}")
    print(f"{'='*60}")

    # Prepare data splits
    X_all = [sequences[i] for i in tr]                          # input features for training
    y_all = [realTrajectoryObject.targets_catch[i] for i in tr] # catch results for training
    X_out = [sequences[i] for i in va]                          # input features for validation
    y_out = [realTrajectoryObject.targets_catch[i] for i in va] # catch results for validation
    
    X_traj = [projTrajectoryObject.sequences[i] for i in va]    # input features based on projected traj for validation

    # Scale features
    scaler = StandardScaler()
    scaler.fit(np.vstack([s for s in X_all]))

    X_all_sc = [scaler.transform(s) for s in X_all]
    X_out_sc = [scaler.transform(s) for s in X_out]
    X_traj_sc = [scaler.transform(s) for s in X_traj]

    # Train LightGBM with GridSearchCV + early stopping
    model, val_auc, val_bce = train_lightgbm_with_tuning(
        X_all_sc, y_all, X_out_sc, y_out, config
    )

    # Predict on REAL trajectories (validation set)
    X_out_flat = np.vstack([s.flatten() for s in X_out_sc])
    preds_out = model.predict(X_out_flat)
    print(f"✓ Predicted on {len(preds_out)} validation plays (real trajectories)")
    print(f"  Mean catch prob: {preds_out.mean():.3f}, Std: {preds_out.std():.3f}")
    
    if want_out_fold_metrics:
        metrics_real = compute_metrics(np.array(y_out), preds_out, prefix="real_")
        print_metrics(metrics_real, title="📊 Validation Metrics (Real Trajectories)")

    # Predict on PROJECTED trajectories
    X_traj_flat = np.vstack([s.flatten() for s in X_traj_sc])
    preds_traj = model.predict(X_traj_flat)
    print(f"✓ Predicted on {len(preds_traj)} projected trajectories")

    # Save catch probability predictions
    game_ids = [projTrajectoryObject.sequence_ids[i]['game_id'] for i in va]
    play_ids = [projTrajectoryObject.sequence_ids[i]['play_id'] for i in va]
    out_df = pd.DataFrame({'game_id': game_ids,'play_id': play_ids, 'pred_catch_prob_by_proj_traj': preds_traj,'pred_catch_prob_by_real_traj': preds_out})
    out_df.to_csv(config.OUTPUT_DIR / 'model_outputs' / CATCH_PROB_OUTPUT_FILE, index=False,  header=(fold == 1),  mode='w' if fold == 1 else 'a') 


    # ===================
    # 🔥 THE COOL PART: Defender Impact Analysis
    # ===================
    if not CALCULATE_DEFENDER_IMPACT:
        continue
    
    val_game_ids = [projTrajectoryObject.sequence_ids[i]['game_id'] for i in va]
    val_play_ids = [projTrajectoryObject.sequence_ids[i]['play_id'] for i in va]
    
    all_impacts = []
    for num, (game_id, play_id) in enumerate(zip(val_game_ids, val_play_ids)):
        if num % 100 == 0:
            print(f"Processing defender impact for game {game_id}, play {play_id} (index {num})")
        
        impact_df = get_defender_impact_flipped(
            model, scaler, 
            projTrajectoryObject.input_df,  # Projected trajectories
            realTrajectoryObject.input_df,   # Real trajectories
            game_id, play_id, config
        )
        
        if impact_df is not None:
            impact_df['game_id'] = game_id
            impact_df['play_id'] = play_id
            impact_df['fold'] = fold
            impact_df = impact_df[['game_id', 'play_id', 'nfl_id', 'player_role', 'baseline_prob', 'real_prob', 'delta', 'fold' ]]
            all_impacts.append(impact_df)
    
    # Save defender impact results
    if all_impacts:
        fold_impact_df = pd.concat(all_impacts, ignore_index=True)
        fold_impact_df.to_csv(config.OUTPUT_DIR / 'model_outputs' / DEFENDER_IMPACT_OUTPUT_FILE, index=False, header=(fold == 1), mode='w' if fold == 1 else 'a'
        )
        print(f"✓ Saved defender impact for fold {fold} with {len(fold_impact_df)} records to {DEFENDER_IMPACT_OUTPUT_FILE}")


# ===================
# Final Summary
# ===================
print("\n" + "="*80)
print("FINAL CROSS-VALIDATION SUMMARY (LightGBM)")
print("="*80)
fold_df = pd.DataFrame(fold_metrics)
print(fold_df.to_string(index=False))
print(f"\nMean AUC: {fold_df['auc'].mean():.4f} ± {fold_df['auc'].std():.4f}")
print(f"Mean BCE: {fold_df['bce'].mean():.4f} ± {fold_df['bce'].std():.4f}")


[3/4] Training LightGBM models...

Fold 1/5
  Training: (10516, 750) | Validation: (2616, 750)
  ⚡ Running GridSearchCV...
Fitting 3 folds for each of 1 candidates, totalling 3 fits


/Users/kniu91/Documents/projects/bdb-26/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/kniu91/Documents/projects/bdb-26/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/kniu91/Documents/projects/bdb-26/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


  ✓ Best CV AUC: -0.4161
  ✓ Best params: {'colsample_bytree': 0.8, 'learning_rate': 0.0075, 'min_child_samples': 20, 'n_estimators': 500, 'num_leaves': 15, 'reg_alpha': 0, 'reg_lambda': 2.0, 'subsample': 0.8}
[500]	train's auc: 0.89657	valid's auc: 0.868032

  📊 Final Metrics:
     Train AUC: 0.8965 | BCE: 0.3744
     Val   AUC: 0.8680 | BCE: 0.3977
✓ Predicted on 2616 validation plays (real trajectories)
  Mean catch prob: 0.683, Std: 0.274

📊 Validation Metrics (Real Trajectories):
  real_bce: 0.3977
  real_auc: 0.8680
  real_acc: 0.8333
✓ Predicted on 2616 projected trajectories
Processing defender impact for game 2023091000, play 185 (index 0)


Processing defender impact for game 2023091003, play 2571 (index 100)


Processing defender impact for game 2023091704, play 2004 (index 200)


...

Processing defender impact for game 2023123107, play 1343 (index 2300)


Processing defender impact for game 2023123111, play 2307 (index 2400)


Processing defender impact for game 2024010709, play 412 (index 2500)


Processing defender impact for game 2024010712, play 185 (index 2600)


✓ Saved defender impact for fold 5 with 8756 records

FINAL CROSS-VALIDATION SUMMARY (LightGBM)
 fold      auc      bce
    1 0.868041 0.397694
    2 0.864369 0.399491
    3 0.857106 0.414629
    4 0.862843 0.416904
    5 0.857583 0.429626

Mean AUC: 0.8620 ± 0.0046
Mean BCE: 0.4117 ± 0.0132
